In [47]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from sklearn.linear_model import LinearRegression
from tinyconformal.utils import NewsvendorSolver

In [48]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

In [49]:
# Financial parameters
selling_price = 320      # Unit selling price
unit_cost = 90         # Unit acquisition cost
annual_holding_rate = 0.14 # Annual capital holding rate (12% p.a.)

days_obsoletes = 180

# 1. Underage Cost (Cu): Lost margin per unfulfilled unit
c_u = selling_price - unit_cost

# 2. Overage Cost (Co): Holding cost + daily obsolescence rate
daily_obsolescence_cost = unit_cost / days_obsoletes
daily_holding_cost = (unit_cost * annual_holding_rate) / 365

estimated_holding_days = 30
c_o = (daily_holding_cost + daily_obsolescence_cost) * estimated_holding_days

# RandomForestQuantileRegressor

In [50]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [51]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
        "LinearRegression": LinearRegression()
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     interval_pairs=[
         ("RF-lo-90", "RF-hi-90"),
     ],
     median_cols="RF-50", # Whether to inter
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,interval_pairs,"[('RF-lo-90', ...)]"
,median_cols,'RF-50'
,n_windows,7
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [52]:
preds = cqr.predict_interval(h=12, X_df=test)

In [53]:
from typing import List, Union

def economic_loss(
    df: pd.DataFrame,
    models: List[str],
    id_col: str = "unique_id",
    target_col: str = "y",
    cost_understock: Union[str, float] = "cu",
    cost_overstock: Union[str, float] = "co",
) -> pd.DataFrame:
    """Calculate Total Economic Loss (financial cost in currency) for multiple models.

    Evaluates forecasting models based on the Newsvendor cost model, measuring
    the financial impact of stockouts (understocking) and excess inventory (overstocking)
    per SKU.

    Parameters
    ----------
    df : pd.DataFrame
        Evaluation DataFrame containing ground-truth values, model forecasts,
        and unit costs for understock and overstock.
    models : List[str]
        List of column names corresponding to the forecasting models to evaluate.
    id_col : str, default="unique_id"
        Column name identifying unique series or group identifiers.
    target_col : str, default="y"
        Column name containing actual target values.
    cost_understock : str or float, default="cu"
        Column name or fixed scalar value representing the unit cost of stockout.
    cost_overstock : str or float, default="co"
        Column name or fixed scalar value representing the unit cost of excess.

    Returns
    -------
    pd.DataFrame
        A DataFrame formatted with `id_col`, `metric` label ('economic_loss'), and
        columns for each evaluated model containing their respective financial losses.

    Notes
    -----
    Interpretation:
    - Measures financial penalty in monetary units (e.g., currency).
    - Calculated as: (Understock Units * Unit Cost of Understock) + (Overstock Units * Unit Cost of Overstock).
    - Always non-negative (>= 0). Lower values indicate higher business efficiency.

    Understock and Overstock Mechanics:
    - **Understock (Rupture):** Occurs when actual demand exceeds the model's forecast (y > forecast).
      Calculated as max(0, y - forecast), representing the units of unmet demand.
    - **Overstock (Excess):** Occurs when the model's forecast exceeds actual demand (forecast > y).
      Calculated as max(0, forecast - y), representing the unsold units left in inventory.

    Final Economic Loss Calculation:
    - For each SKU and period, the total monetary loss is obtained by weighting
      the shortage and excess units by their respective unit costs (C_u and C_o):
      Economic Loss = (Understock × C_u) + (Overstock × C_o)
    - Finally, these individual period losses are aggregated by summing them over time
      for each unique SKU (`id_col`).
    """
    if not models:
        raise ValueError("The 'models' list cannot be empty.")
    if not all(model in df.columns for model in models):
        missing_models = [model for model in models if model not in df.columns]
        raise ValueError(
            f"The following model columns are missing from the DataFrame: {missing_models}"
        )

    cu = (
        df[cost_understock]
        if isinstance(cost_understock, str) and cost_understock in df.columns
        else cost_understock
    )
    co = (
        df[cost_overstock]
        if isinstance(cost_overstock, str) and cost_overstock in df.columns
        else cost_overstock
    )

    understock = df[models].rsub(df[target_col], axis=0).clip(lower=0)
    overstock = df[models].sub(df[target_col], axis=0).clip(lower=0)
    loss_per_row = round((understock.mul(cu, axis=0)) + (overstock.mul(co, axis=0)), 2)
    res = loss_per_row.groupby(df[id_col], observed=True).sum().reset_index()

    res.insert(1, "metric", "economic_loss")
    return res

In [54]:
preds.loc[:,"cu"] = float(c_u)
preds.loc[:,"co"] = float(c_o)

In [55]:
res = NewsvendorSolver.optimize(
    preds, 
    interval_pair=("RF-lo-90-cqr", "RF-hi-90-cqr"), 
    level=90, 
    median_col="RF-50", 
    cost_overstock="co", 
    cost_understock="cu",
    )
res.loc[:, "y"] = test["y"].values

In [56]:
economic_loss(res, ["RF-50", "RF-50-cqr", "RF-lo-90-cqr", "RF-hi-90-cqr", "LinearRegression", "y_optimal"], id_col="unique_id")

unique_id         metric     RF-50  RF-lo-90-cqr  RF-hi-90-cqr  \
0         1  economic_loss  277949.5      629590.5       17159.7   

   LinearRegression  y_optimal  
0         124410.89    16182.7

In [57]:
from typing import List, Union
import numpy as np
import pandas as pd

def tail_risk(
    df: pd.DataFrame,
    models: List[str],
    id_col: str = "unique_id",
    target_col: str = "y",
    cost_understock: Union[str, float] = "cu",
    cost_overstock: Union[str, float] = "co",
    risk_level: float = 0.95,
) -> pd.DataFrame:
    """Calculate financial risk and tail-risk metrics for multiple models.

    Computes Expected Cost, Standard Deviation, Value at Risk (VaR), 
    Conditional Value at Risk (CVaR), and Worst Scenario on item-time 
    level losses using a vectorized approach.

    Parameters
    ----------
    df : pd.DataFrame
        Evaluation DataFrame containing ground-truth values and model forecasts.
    models : List[str]
        List of column names corresponding to the forecasting models to evaluate.
    id_col : str, default="unique_id"
        Column name identifying unique series or group identifiers.
    target_col : str, default="y"
        Column name containing actual target values.
    cost_understock : Union[str, float], default="cu"
        The unit cost of understocking (c_u). Can be a scalar or a column name.
    cost_overstock : Union[str, float], default="co"
        The unit cost of overstocking (c_o). Can be a scalar or a column name.
    risk_level : float, default=0.95
        The statistical risk/confidence level used to define the tail cutoff percentile.

    Returns
    -------
    pd.DataFrame
        A DataFrame structured with `id_col`, `metric` label ('expected_cost', 
        'std_dev', 'var', 'cvar', 'worst_scenario'), and columns for each 
        evaluated model containing their respective values.
    """
    if not models:
        raise ValueError("The 'models' list cannot be empty.")
    if not all(model in df.columns for model in models):
        missing_models = [model for model in models if model not in df.columns]
        raise ValueError(
            f"The following model columns are missing from the DataFrame: {missing_models}"
        )

    y_true = df[target_col].to_numpy()

    c_u = (
        df[cost_understock].to_numpy()
        if isinstance(cost_understock, str) and cost_understock in df.columns
        else cost_understock
    )
    c_o = (
        df[cost_overstock].to_numpy()
        if isinstance(cost_overstock, str) and cost_overstock in df.columns
        else cost_overstock
    )

    preds = df[models].to_numpy()
    understock = np.maximum(0, y_true[:, None] - preds)
    overstock = np.maximum(0, preds - y_true[:, None])

    cu_arr = c_u[:, None] if isinstance(c_u, np.ndarray) else c_u
    co_arr = c_o[:, None] if isinstance(c_o, np.ndarray) else c_o

    losses_matrix = (understock * cu_arr) + (overstock * co_arr)
    loss_df = pd.DataFrame(losses_matrix, columns=models, index=df[id_col])

    ec_results = []
    std_results = []
    var_results = []
    cvar_results = []
    worst_results = []

    grouped = loss_df.groupby(level=0, observed=True)

    for item_id, group in grouped:
        ec_row = {id_col: item_id}
        std_row = {id_col: item_id}
        var_row = {id_col: item_id}
        cvar_row = {id_col: item_id}
        worst_row = {id_col: item_id}
        
        arr_group = group.to_numpy()

        for m_idx, model in enumerate(models):
            model_losses = arr_group[:, m_idx]
            clean_losses = model_losses[~np.isnan(model_losses)]

            if clean_losses.size == 0:
                ec_row[model] = np.nan
                std_row[model] = np.nan
                var_row[model] = np.nan
                cvar_row[model] = np.nan
                worst_row[model] = np.nan
                continue

            ec_val = clean_losses.mean()
            std_val = clean_losses.std(ddof=1) if clean_losses.size > 1 else 0.0
            var_val = np.quantile(clean_losses, risk_level, method="higher")
            
            tail_mask = clean_losses >= var_val
            cvar_val = clean_losses[tail_mask].mean() if tail_mask.any() else var_val
            worst_val = clean_losses.max()

            ec_row[model] = round(float(ec_val), 2)
            std_row[model] = round(float(std_val), 2)
            var_row[model] = round(float(var_val), 2)
            cvar_row[model] = round(float(cvar_val), 2)
            worst_row[model] = round(float(worst_val), 2)

        ec_results.append(ec_row)
        std_results.append(std_row)
        var_results.append(var_row)
        cvar_results.append(cvar_row)
        worst_results.append(worst_row)

    df_ec = pd.DataFrame(ec_results)
    df_ec.insert(1, "metric", "expected_cost")

    df_std = pd.DataFrame(std_results)
    df_std.insert(1, "metric", "std_dev")

    df_var = pd.DataFrame(var_results)
    df_var.insert(1, "metric", "var")

    df_cvar = pd.DataFrame(cvar_results)
    df_cvar.insert(1, "metric", "cvar")

    df_worst = pd.DataFrame(worst_results)
    df_worst.insert(1, "metric", "worst_scenario")

    res = pd.concat([df_ec, df_std, df_var, df_cvar, df_worst], ignore_index=True)
    return res

In [58]:
tail_risk(res, ["RF-50", "RF-50-cqr", "RF-lo-90-cqr", "RF-hi-90-cqr", "LinearRegression", "y_optimal"], id_col="unique_id")

,unique_id,metric,RF-50,RF-50-cqr,RF-lo-90-cqr,RF-hi-90-cqr,LinearRegression,y_optimal
0,1,expected_cost,23162.46,24901.38,52465.88,1429.98,10367.58,1363.39
1,1,std_dev,20037.38,21867.83,23990.52,928.68,15456.76,882.55
2,1,var,59570.00,65867.64,95887.00,2918.48,41571.88,2801.63
3,1,cvar,59570.00,65867.64,95887.00,2918.48,41571.88,2801.63
4,1,worst_scenario,59570.00,65867.64,95887.00,2918.48,41571.88,2801.63
